# 05 — Results Analysis
**Deep Learning for Cryptocurrency Price Forecasting**
*Muluh Penn Junior Patrick — M.Tech. Thesis 2026*

---
Comprehensive analysis of all experimental results:
- Model comparison table (all 6 models, all KPIs)
- Statistical significance (Diebold-Mariano test matrix)
- Price prediction vs actual plots
- Feature ablation study results
- Discussion of findings


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({'figure.dpi': 150, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True,
                     'grid.alpha': 0.3, 'savefig.dpi': 300})
RESULTS_DIR = Path('../experiments/results')
FIGS_DIR    = Path('../experiments/figures')
print('Environment ready.')


## 5.1 Primary results table — BTC 1d h=1

In [ ]:
from src.evaluation.evaluator import load_all_results, print_comparison_table

# Print formatted comparison table
print_comparison_table(asset='BTC', interval='1d', horizon=1)


## 5.2 Overall model rankings

In [ ]:
from src.evaluation.evaluator import rank_models

ranked = rank_models(asset='BTC', interval='1d', horizon=1)
if not ranked.empty:
    print('\nModel Rankings (lower avg_rank = better overall):')
    print(ranked.to_string())


## 5.3 Diebold-Mariano significance matrix

In [ ]:
import numpy as np
from src.evaluation.diebold_mariano import load_errors_from_results, print_dm_results

errors = load_errors_from_results(asset='BTC', interval='1d', horizon=1)
if len(errors) >= 2:
    print_dm_results(errors, h=1, alpha=0.05)
else:
    print('Need at least 2 models with saved predictions. Run save_predictions.py first.')


## 5.4 Price prediction plot — GRU model

In [ ]:
# Load GRU predictions
pred_path  = RESULTS_DIR / 'gru_BTC_1d_h1_predictions.npy'
close_path = Path('../data/processed/scalers/BTC_1d_test_close.npy')

if pred_path.exists() and close_path.exists():
    errors     = np.load(pred_path)
    test_close = np.load(close_path)
    n          = len(errors)
    seq_len    = 90
    idx        = np.clip(np.arange(n) + seq_len - 1, 0, len(test_close) - 1)
    actuals    = test_close[idx]
    preds      = actuals - errors
    dates      = pd.date_range('2024-12-26', periods=n, freq='D')

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7),
                                    gridspec_kw={'height_ratios': [3, 1]})
    ax1.plot(dates, actuals/1000, color='#1a1a2e', linewidth=1.5, label='Actual BTC price')
    ax1.plot(dates, preds/1000, color='#1D9E75', linewidth=1.2, linestyle='--',
             label=f'GRU prediction (MAPE={np.mean(np.abs(errors/actuals))*100:.2f}%)', alpha=0.9)
    ax1.fill_between(dates, actuals/1000, preds/1000, alpha=0.07, color='#1D9E75')
    ax1.set_ylabel('Price (USD thousands)'); ax1.legend()
    ax1.set_title('GRU Price Prediction vs Actual — Test Period Dec 2024–Mar 2026')
    ax1.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))

    colors = ['#E24B4A' if e < 0 else '#1D9E75' for e in errors]
    ax2.bar(dates, errors/1000, color=colors, width=0.8, alpha=0.7)
    ax2.axhline(0, color='black', linewidth=0.8)
    ax2.set_ylabel('Error (k USD)'); ax2.set_xlabel('Date')
    ax2.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
    plt.tight_layout()
    plt.savefig(FIGS_DIR / 'fig2_price_prediction_gru_notebook.png', bbox_inches='tight')
    plt.show()
else:
    print('Prediction arrays not found. Run save_predictions.py first.')


## 5.5 Ablation study results

In [ ]:
ablation_path = RESULTS_DIR / 'ablation_lstm_BTC_1d_h1.csv'
if ablation_path.exists():
    df = pd.read_csv(ablation_path)
    baseline_mape = float(df.loc[df['condition']=='full', 'mape'].values[0])

    fig, ax = plt.subplots(figsize=(8, 5))
    ablation = df[df['condition'] != 'full'].copy()
    ablation['delta'] = ablation['mape'] - baseline_mape
    colors = ['#E24B4A' if d > 0 else '#1D9E75' for d in ablation['delta']]
    ax.barh(ablation['condition'], ablation['delta'], color=colors, height=0.5, alpha=0.85)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('ΔMAPE vs full model (positive = worse without this group)')
    ax.set_title(f'Feature Ablation Study — LSTM BTC 1d h=1 (baseline MAPE: {baseline_mape:.2f}%)')
    plt.tight_layout(); plt.show()
    print(df[['condition','mape','rmse']].to_string(index=False))
else:
    print('Ablation results not found. Run: python -m src.evaluation.ablation_study --model lstm --asset BTC')


## 5.6 MAPE comparison across all models

In [ ]:
models  = ['lstm','gru','bilstm','cnn_lstm','attention_lstm','transformer']
labels  = ['LSTM','GRU','BiLSTM','CNN-LSTM','Attn-LSTM','Transformer']
colors  = ['#3266AD','#1D9E75','#D85A30','#BA7517','#7F77DD','#888780']
mapes, rmses = [], []

for model in models:
    path = RESULTS_DIR / f'{model}_BTC_1d_h1_results.csv'
    if path.exists():
        df = pd.read_csv(path)
        mapes.append(float(df.iloc[0]['mape']))
        rmses.append(float(df.iloc[0]['rmse']))
    else:
        mapes.append(float('nan'))
        rmses.append(float('nan'))

x = range(len(labels))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
bars1 = ax1.bar(x, mapes, color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
ax1.axhline(5.0, color='#E24B4A', linestyle='--', linewidth=1, label='5% target', alpha=0.7)
ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=20, ha='right')
ax1.set_ylabel('MAPE (%)'); ax1.set_title('MAPE (lower is better)'); ax1.legend()
for bar, v in zip(bars1, mapes):
    if not np.isnan(v):
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                 f'{v:.2f}%', ha='center', va='bottom', fontsize=8)

bars2 = ax2.bar(x, [r/1000 for r in rmses], color=colors, alpha=0.85,
                edgecolor='white', linewidth=0.5)
ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=20, ha='right')
ax2.set_ylabel('RMSE (USD thousands)'); ax2.set_title('RMSE (lower is better)')
for bar, v in zip(bars2, rmses):
    if not np.isnan(v):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                 f'${v/1000:.1f}k', ha='center', va='bottom', fontsize=8)

plt.suptitle('Model Performance — BTC Daily h=1 | Test: Dec 2024–Mar 2026',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'fig3_model_comparison_notebook.png', bbox_inches='tight')
plt.show()


## 5.7 Key findings summary

| Finding | Evidence |
|---------|----------|
| GRU ≈ LSTM (not significant) | DM = −0.285, p > 0.05 |
| GRU > Transformer (significant) | DM = −3.103*, p < 0.05 |
| BiLSTM worse than all | DM = +4.8 to +8.4*, p < 0.05 across all pairs |
| All models beat MAPE < 5% | Best: GRU at 1.60% |
| Feature ablation shows no group is critical | ΔMAPE all < 0.01% at h=1 |

**Conclusion:** For 1-day BTC forecasting, recurrent architectures (GRU, LSTM) with
modest capacity outperform the more complex Transformer. BiLSTM's causal violation
(using future information in backward pass) is a practical concern for trading applications.
Feature engineering beyond OHLCV+TA contributes minimally at h=1 — likely more impactful
at longer horizons.

**→ Next: Visualization Gallery** (`06_visualization_gallery.ipynb`)
